# Ejercicio 4: Splicing Alternativo y Extensión con Biopython / Ensembl

## 📖 Introducción y Fundamento Teórico Resumido

> **Objetivo:** Comprender cómo un mismo gen puede generar múltiples proteínas diferentes mediante el procesamiento alternativo del pre-ARNm, analizar sus implicaciones estructurales y estudiar el caso real en Ensembl del receptor humano **FGFR2**.

### Resumen de los puntos teóricos del enunciado:

1. **Estructura génica de partida (5 exones):**
   Un gen eucariota típico se transcribe en un pre-ARNm con exones codificantes separados por intrones:
   $$\text{pre-ARNm:} \quad 5'\text{- [Exón 1] – (Intrón 1) – [Exón 2] – (Intrón 2) – [Exón 3] – (Intrón 3) – [Exón 4] – (Intrón 4) – [Exón 5] -}3'$$

2. **Diseño de combinaciones de splicing alternativo:**
   - **Isoforma Constitutiva (1-2-3-4-5):** Incluye todos los exones; representa la proteína completa canónica.
   - **Isoforma con salto de exón (*Exon Skipping*, 1-2-4-5):** Excluye selectivamente el Exón 3.
   - **Isoforma con exclusión modular (1-3-5):** Excluye simultáneamente los exones conectores 2 y 4.

3. **Diferencias funcionales en las proteínas resultantes:**
   - Cada exón suele codificar un **dominio estructural o funcional discreto** (*exon shuffling*).
   - **Exón 3:** Si codifica el dominio catalítico o de unión a ligando, la isoforma **1-2-4-5** generará una proteína afuncional o con efecto **dominante-negativo** (puede anclarse o dimerizar pero no cataliza ni transmite señal).
   - **Exones 2 y 4:** Si actúan como bisagras o espaciadores reguladores, la isoforma **1-3-5** producirá una proteína compacta y rígida, pudiendo presentar activación constitutiva o alteración de estabilidad.
   - **Fase de lectura (*in-frame*):** Si la suma de nucleótidos de los exones excluidos es múltiplo de 3, la lectura se mantiene en fase. Si no lo es (*frameshift*), se produce un codón de parada prematuro y el transcrito es destruido por la vía de **degradación mediada por mutaciones sin sentido (NMD)**.

4. **Reflexión: Diversidad del proteoma sin aumento de genes:**
   - **La paradoja de los genes:** El genoma humano cuenta solo con $\sim 20.000$ genes codificantes (cifra similar a la del nematodo *C. elegans*), pero produce más de **$100.000$ transcritos maduros** y hasta **$1.000.000$ de proteoformas** funcionales.
   - **Ventajas biológicas:** Máxima economía de espacio y energía en el ADN cromosómico, modularidad evolutiva y regulación precisa según el tejido o la fase de desarrollo.

# 💻 Extensión con Biopython / Ensembl: Guía Paso a Paso

> **Enunciado de la extensión:**
> *Busca en Ensembl un gen humano conocido con isoformas (por ejemplo, FGFR2) y compara sus diferentes transcritos. Reflexiona sobre cómo estas diferencias podrían afectar a la función de la proteína.*

A continuación, modelaremos computacionalmente las combinaciones de splicing y analizaremos el caso real de Ensembl del receptor **FGFR2**.

### Paso 1: Modelado computacional del gen de 5 exones con Biopython

Para simular el splicing en Python, definimos 5 secuencias discretas de exones en fase como objetos `Seq`:
- **Exón 1:** Péptido señal y codón de inicio `ATG`.
- **Exón 2:** Dominio regulador N-terminal.
- **Exón 3:** Dominio catalítico / sitio de unión a ligando.
- **Exón 4:** Dominio espaciador / bisagra flexible.
- **Exón 5:** Dominio C-terminal y señal de parada `TAA`.

Ejecutemos la siguiente celda para inicializar y traducir los exones individuales:

In [1]:
from Bio.Seq import Seq

# Definición de 5 exones hipotéticos en fase de lectura (múltiplos de 3)
exones = {
    "Exon_1": Seq("ATGGCTAGCAAC"),            # Met-Ala-Ser-Asn (Inicio / Señal)
    "Exon_2": Seq("GTCCTGCTGGAC"),            # Val-Leu-Leu-Asp (Regulador A)
    "Exon_3": Seq("AAGTGCCACTACGAG"),         # Lys-Cys-His-Tyr-Glu (Sitio Activo / Unión)
    "Exon_4": Seq("TTCAACGGCAAGATC"),         # Phe-Asn-Gly-Lys-Ile (Bisagra flexible)
    "Exon_5": Seq("CCCTGGCTCTACCGCTAA"),     # Pro-Trp-Leu-Tyr-Arg-STOP (C-term / Parada)
}

print("=" * 65)
print("EXONES INDIVIDUALES DEL PRE-ARNm")
print("=" * 65)
for nombre, seq in exones.items():
    print(f"{nombre} ({len(seq)} nt): {seq} -> {seq.translate()}")
print("=" * 65)

EXONES INDIVIDUALES DEL PRE-ARNm
Exon_1 (12 nt): ATGGCTAGCAAC -> MASN
Exon_2 (12 nt): GTCCTGCTGGAC -> VLLD
Exon_3 (15 nt): AAGTGCCACTACGAG -> KCHYE
Exon_4 (15 nt): TTCAACGGCAAGATC -> FNGKI
Exon_5 (18 nt): CCCTGGCTCTACCGCTAA -> PWLYR*


### Paso 2: Ensamblado y traducción de las isoformas de splicing alternativo

En Biopython, concatenar exones para simular el corte y empalme del espliceosoma es tan directo como sumar objetos `Seq` (`exon1 + exon2 + ...`).

Construimos las tres variantes solicitadas:
1. **Isoforma Constitutiva:** `1 + 2 + 3 + 4 + 5`
2. **Isoforma B (Salto de Exón 3):** `1 + 2 + 4 + 5`
3. **Isoforma C (Exclusión modular):** `1 + 3 + 5`

In [2]:
# Combinaciones de splicing alternativo mediante concatenación de Seq
isoformas_mrna = {
    "Isoforma A (Constitutiva 1-2-3-4-5)": (
        exones["Exon_1"] + exones["Exon_2"] + exones["Exon_3"] + exones["Exon_4"] + exones["Exon_5"]
    ),
    "Isoforma B (Salto de Exón 3: 1-2-4-5)": (
        exones["Exon_1"] + exones["Exon_2"] + exones["Exon_4"] + exones["Exon_5"]
    ),
    "Isoforma C (Exclusión Exones 2 y 4: 1-3-5)": (
        exones["Exon_1"] + exones["Exon_3"] + exones["Exon_5"]
    ),
}

print("=" * 70)
print("TRADUCCIÓN DE ISOFORMAS DE SPLICING ALTERNATIVO")
print("=" * 70)
for nombre, arnm in isoformas_mrna.items():
    proteina = arnm.translate(to_stop=True)
    print(f"\n* {nombre}:")
    print(f"  Longitud ARNm:      {len(arnm)} nucleótidos")
    print(f"  Secuencia ARNm:     5'- {arnm} -3'")
    print(f"  Proteína ({len(proteina)} aa):  {proteina}")
print("=" * 70)

TRADUCCIÓN DE ISOFORMAS DE SPLICING ALTERNATIVO

* Isoforma A (Constitutiva 1-2-3-4-5):
  Longitud ARNm:      72 nucleótidos
  Secuencia ARNm:     5'- ATGGCTAGCAACGTCCTGCTGGACAAGTGCCACTACGAGTTCAACGGCAAGATCCCCTGGCTCTACCGCTAA -3'
  Proteína (23 aa):  MASNVLLDKCHYEFNGKIPWLYR

* Isoforma B (Salto de Exón 3: 1-2-4-5):
  Longitud ARNm:      57 nucleótidos
  Secuencia ARNm:     5'- ATGGCTAGCAACGTCCTGCTGGACTTCAACGGCAAGATCCCCTGGCTCTACCGCTAA -3'
  Proteína (18 aa):  MASNVLLDFNGKIPWLYR

* Isoforma C (Exclusión Exones 2 y 4: 1-3-5):
  Longitud ARNm:      45 nucleótidos
  Secuencia ARNm:     5'- ATGGCTAGCAACAAGTGCCACTACGAGCCCTGGCTCTACCGCTAA -3'
  Proteína (14 aa):  MASNKCHYEPWLYR


### Paso 3: Análisis del impacto funcional en las proteínas del modelo

Mostramos una tabla comparativa resumiendo la longitud, los dominios presentes y el impacto fisiológico de cada isoforma:

In [3]:
print("=" * 75)
print(f"{'Isoforma':<32} | {'Long. (aa)':<10} | {'Secuencia':<20} | {'Efecto funcional'}")
print("-" * 75)
for nombre, arnm in isoformas_mrna.items():
    prot = arnm.translate(to_stop=True)
    nombre_corto = nombre.split(" (")[0]
    if "Constitutiva" in nombre:
        efecto = "Canónica completa"
    elif "Salto" in nombre:
        efecto = "Pierde sitio activo (KCHYE) -> Dominante-negativa"
    else:
        efecto = "Compacta / Pérdida de bisagras reguladoras"
    print(f"{nombre_corto:<32} | {len(prot):<10} | {str(prot):<20} | {efecto}")
print("=" * 75)

Isoforma                         | Long. (aa) | Secuencia            | Efecto funcional
---------------------------------------------------------------------------
Isoforma A                       | 23         | MASNVLLDKCHYEFNGKIPWLYR | Canónica completa
Isoforma B                       | 18         | MASNVLLDFNGKIPWLYR   | Pierde sitio activo (KCHYE) -> Dominante-negativa
Isoforma C                       | 14         | MASNKCHYEPWLYR       | Compacta / Pérdida de bisagras reguladoras


### Paso 4: Caso Real de Ensembl — El Receptor Humano *FGFR2* (`ENSG00000066468`)

El gen **FGFR2** (*Fibroblast Growth Factor Receptor 2*) es un ejemplo canónico de splicing alternativo con **exones mutuamente excluyentes**:

```text
Locus FGFR2 pre-ARNm:
  ───[Exón 7]───┬───[Exón 8 / IIIb]───┬───[Exón 9 / IIIc]───┬───[Exón 10]───
                │                     │                     │
    Epitelio    └─────────────────────┴──────(Excluido)─────┘ ───> Isoforma FGFR2-IIIb (ENST00000358487)
                │                     │                     │
    Mesénquima  └─────(Excluido)──────┴─────────────────────┘ ───> Isoforma FGFR2-IIIc (ENST00000356226)
```

Cargamos el archivo FASTA real `data/fgfr2_isoformas.fasta` que contiene las secuencias de los exones variables IIIb y IIIc extraídas de Ensembl:

In [4]:
from pathlib import Path
from Bio import SeqIO

# Localización robusta del archivo FASTA
base_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
fasta_path = base_dir / "data" / "fgfr2_isoformas.fasta"

print(f"Cargando secuencias de Ensembl desde: {fasta_path.name}")
records = list(SeqIO.parse(fasta_path, "fasta"))
print(f"Número de transcritos/exones cargados: {len(records)}\n")

for rec in records:
    prot_fragment = rec.seq.translate(to_stop=True)
    print("-" * 65)
    print(f"ID Ensembl:    {rec.id}")
    print(f"Descripción:   {rec.description}")
    print(f"Longitud ADN:  {len(rec.seq)} pb")
    print(f"Secuencia aa:  {prot_fragment}")
    print(f"Longitud aa:   {len(prot_fragment)} residuos")
print("-" * 65)

Cargando secuencias de Ensembl desde: fgfr2_isoformas.fasta
Número de transcritos/exones cargados: 2

-----------------------------------------------------------------
ID Ensembl:    ENST00000358487_FGFR2_IIIb_Exon
Descripción:   ENST00000358487_FGFR2_IIIb_Exon Exon variable IIIb (Epitelial) - Dominio IgIII
Longitud ADN:  135 pb
Secuencia aa:  IESSNKINLNVSFNNVTWLEDAGNYTCLAGNSIGISFHSAWLTVL
Longitud aa:   45 residuos
-----------------------------------------------------------------
ID Ensembl:    ENST00000356226_FGFR2_IIIc_Exon
Descripción:   ENST00000356226_FGFR2_IIIc_Exon Exon variable IIIc (Mesenquimal) - Dominio IgIII
Longitud ADN:  135 pb
Secuencia aa:  FKCPSSGTPNPTLRWLKNGKEFKPDHRIGGYKVRYATWSIIMDSV
Longitud aa:   45 residuos
-----------------------------------------------------------------


### Paso 5: Repercusión Biológica, Especificidad de Ligando y Cáncer (EMT)

Analicemos cómo la elección de un solo exón u otro cambia completamente la fisiología celular y su papel patológico:

In [5]:
print("=" * 75)
print("COMPARACIÓN BIOLÓGICA Y MÉDICA DE LAS ISOFORMAS DE FGFR2 (ENSEMBL)")
print("=" * 75)

print("1. Isoforma Epitelial (FGFR2-IIIb - ENST00000358487 / Exón 8):")
print("   * Expresión tisular:      Restringida a células EPITELIALES.")
print("   * Afinidad por ligandos:  Se une fuertemente a FGF7 (KGF) y FGF10.")
print("   * Función fisiológica:    Mantenimiento y homeostasis de epitelios en piel y pulmón.")

print("\n2. Isoforma Mesenquimal (FGFR2-IIIc - ENST00000356226 / Exón 9):")
print("   * Expresión tisular:      Restringida a células MESENQUIMALES.")
print("   * Afinidad por ligandos:  Se une fuertemente a FGF2 (bFGF) y FGF4.")
print("   * Función fisiológica:    Morfogénesis mesenquimal y osteogénesis embrionaria.")

print("\n3. Importancia en Oncología: Transición Epitelio-Mesénquima (EMT):")
print("   * La conmutación de splicing de la isoforma IIIb a la IIIc es un evento")
print("     patológico capital durante la progresión del cáncer (carcinomas).")
print("   * Al adquirir la isoforma IIIc, la célula tumoral responde a factores autocrinos")
print("     (FGF2), disuelve las uniones intercelulares y adquiere motilidad, invasividad")
print("     y capacidad metastásica hacia otros tejidos.")
print("=" * 75)

COMPARACIÓN BIOLÓGICA Y MÉDICA DE LAS ISOFORMAS DE FGFR2 (ENSEMBL)
1. Isoforma Epitelial (FGFR2-IIIb - ENST00000358487 / Exón 8):
   * Expresión tisular:      Restringida a células EPITELIALES.
   * Afinidad por ligandos:  Se une fuertemente a FGF7 (KGF) y FGF10.
   * Función fisiológica:    Mantenimiento y homeostasis de epitelios en piel y pulmón.

2. Isoforma Mesenquimal (FGFR2-IIIc - ENST00000356226 / Exón 9):
   * Expresión tisular:      Restringida a células MESENQUIMALES.
   * Afinidad por ligandos:  Se une fuertemente a FGF2 (bFGF) y FGF4.
   * Función fisiológica:    Morfogénesis mesenquimal y osteogénesis embrionaria.

3. Importancia en Oncología: Transición Epitelio-Mesénquima (EMT):
   * La conmutación de splicing de la isoforma IIIb a la IIIc es un evento
     patológico capital durante la progresión del cáncer (carcinomas).
   * Al adquirir la isoforma IIIc, la célula tumoral responde a factores autocrinos
     (FGF2), disuelve las uniones intercelulares y adquiere motili

### Paso 6 (Extra interactivo): Simulador modular de splicing alternativo

Definimos una función interactiva `simular_splicing(combinacion_exones)` que permite ensamblar cualquier combinación de los exones disponibles (por ejemplo, `['Exon_1', 'Exon_4', 'Exon_5']` o `['Exon_1', 'Exon_2', 'Exon_3', 'Exon_5']`), verificando automáticamente la longitud y traduciendo la proteína resultante.

In [6]:
def simular_splicing(lista_exones: list[str], diccionario_exones: dict[str, Seq]) -> dict:
    """
    Ensambla una lista ordenada de exones, calcula el transcrito de ARNm
    y obtiene el polipéptido resultante evaluando la fase de lectura.
    """
    # Concatenación de los exones seleccionados
    arnm_maduro = Seq("")
    for nombre in lista_exones:
        if nombre not in diccionario_exones:
            raise KeyError(f"Exón '{nombre}' no encontrado en el diccionario.")
        arnm_maduro += diccionario_exones[nombre]
    
    # Traducción
    proteina = arnm_maduro.translate(to_stop=True)
    en_fase = len(arnm_maduro) % 3 == 0
    
    print(f"\n>>> Combinación de Splicing: {' - '.join(lista_exones)}")
    print(f"    Longitud ARNm:      {len(arnm_maduro)} nt ({'En fase de tripletes' if en_fase else 'Desfase / Frameshift'})")
    print(f"    Péptido resultante: {proteina} ({len(proteina)} aa)")
    
    return {"arnm": arnm_maduro, "proteina": proteina, "en_fase": en_fase}

# Probamos una combinación novedosa: Exón 1 + Exón 3 + Exón 4 + Exón 5 (salto del Exón 2)
res1 = simular_splicing(["Exon_1", "Exon_3", "Exon_4", "Exon_5"], exones)

# Probamos otra combinación mínima: Exón 1 + Exón 5
res2 = simular_splicing(["Exon_1", "Exon_5"], exones)


>>> Combinación de Splicing: Exon_1 - Exon_3 - Exon_4 - Exon_5
    Longitud ARNm:      60 nt (En fase de tripletes)
    Péptido resultante: MASNKCHYEFNGKIPWLYR (19 aa)

>>> Combinación de Splicing: Exon_1 - Exon_5
    Longitud ARNm:      30 nt (En fase de tripletes)
    Péptido resultante: MASNPWLYR (9 aa)


## 📌 Resumen de lo aprendido

- **Splicing alternativo:** Proceso postranscripcional donde diferentes combinaciones de exones se empalman en el ARNm maduro, multiplicando exponencialmente la diversidad de proteínas.
- **Modularidad funcional (*Exon Shuffling*):** Cada exón aporta dominios o propiedades específicas (anclaje, catálisis, especificidad de unión).
- **Mantenimiento del marco de lectura:** Los exones alternativos en fase generan isoformas viables; la alteración del marco induce codones de parada prematuros (PTC) activando la degradación mediada por mutaciones sin sentido (NMD).
- **Caso FGFR2:** El cambio entre exones mutuamente excluyentes (IIIb vs IIIc) altera radicalmente la afinidad por ligandos y marca la Transición Epitelio-Mesénquima (EMT) en la progresión oncológica.